In [5]:
import pandas as pd
import os
import asyncio
import json
import joblib
import threading
import duckdb  # type: ignore
from flask import Flask, jsonify, request
import pandas as pd
import re
from typing import Dict, List, Tuple
from function.colfuzzy import Colfuzzy
from function.insight import Insight
from function.changechart import Changechart
from function.changeinsight import ChangeInsight
from function.nonscientific import NonScientific
from function.checkml import Check
from function.insightmachinelearning import InsightMachineLearning
from function.mlinsightgenerate import MachinelearningInsightGenerate
from function.query import Query
from function.search import SerpApiContentProcessor
from function.changeinsight2 import ChangeInsight2
from function.changeinsight3 import ChangeInsight3
from function.parse2 import Parse2
from function.changeinsight4 import ChangeInsight4
from function.sql_nonambigious import Nonambigioussql
from function.sql_ambigious import Ambigioussql
from function.checkambigious import AmbigiousCheck
from function.regenerate import ReGeneratsql
from function.changeinsight5 import ChangeInsight5
from function.insight2 import Insight2
from sqlglot import exp, parse_one
import copy
import re
import ast


from function.dataanalyze import DataAnalyzer
from function.divide import Divide
from function.documentsearcher import DocumentSearcher
from function.finalquestion import FinalQuestion
from function.likeexact import LikeExact
from function.generatesql import Generatesql
from function.splitunit import Splitunit
from function.match import Match
from function.stringcheck import StringCheckMeaning
from function.textContent import TextContent
from function.visualrecommendation import VisualRecommend
from function.textsynthesis import TextSynthesis
from function.parse import Parse
from function.stringsynthesis import StringSynthesis
from function.documentembedder import DocumentEmbedder
from function.addinformation import AddInformation
from function.brace_toolkit import BraceSQLToolkit

import uuid
import os
import sqlglot
from sqlglot import exp
from function.translate import Translate
from function.phase import Phase
excel_path = "Question_table.xlsx"

df = pd.read_excel(excel_path)

for idx, row in df.iterrows():
    question = row["Human_Query_with_Ambiguity"]
    dataset_name = str(row["Dataset"]) + ".csv"
    dataset_path = os.path.join(os.path.dirname(excel_path), dataset_name)

    print(f"\n=== 第 {idx+1} 行 ===")
    print(f"问题: {question}")
    print(f"数据集文件: {dataset_path}")

    if os.path.exists(dataset_path):
        try:
            data = pd.read_csv(dataset_path)
            print(f"成功读取 {dataset_name}, 行数: {len(data)} 列数: {len(data.columns)}")
            print(data.head())
            file_name=str(row["Dataset"])
            goal=question
            df = data
            df.columns = [col.replace(" ", "_") for col in df.columns]
            AmbigiousCheck_instance = AmbigiousCheck(df, file_name)
            AmbigiousCheck_instance_result = AmbigiousCheck_instance.Check(question=goal)
            print("AmbigiousCheck_instance_result:", AmbigiousCheck_instance_result)
            if AmbigiousCheck_instance_result == 'True':
                Ambigioussql_instance = Ambigioussql(df, file_name)
                print("Ambigioussql_instance:")
                Ambigioussql_instance_result = Ambigioussql_instance.Generatsql(text=goal)
                sql = Ambigioussql_instance_result
            else:
                duckdb_conn = duckdb.connect()
                duckdb_conn.register(file_name, df)

                success = False
                while not success:
                    Nonambigioussql_instance = Nonambigioussql(df, file_name)
                    Nonambigioussql_instance_result = Nonambigioussql_instance.Generatsql(text=goal)
                    sql = Nonambigioussql_instance_result
                    print("生成的 SQL:", sql)
                    try:
                        duckdb_check_result = duckdb_conn.execute(sql).df()
                        success = True
                    except duckdb.Error as e:
                        print(f"SQL执行错误: {e}")


            toolkit = BraceSQLToolkit()

            while True:
                if AmbigiousCheck_instance_result == 'False':
                    break
                print("生成的 SQL:", sql)
                errs = toolkit.invalid_predicates(sql)
                if errs:
                    print("✗ 校验失败！问题位置：")
                    syntax_errors = []
                    for clause, expr in errs:
                        print(f"  - {clause}: {expr}")
                        syntax_errors.append(expr)
                    print(syntax_errors)

                    ReGeneratsql_instance = ReGeneratsql(df, file_name)
                    sql = ReGeneratsql_instance.Generate(
                        text=goal, syntax_error=syntax_errors
                    )
                    if not(toolkit.has_placeholder(sql)):
                        duckdb_conn = duckdb.connect()
                        duckdb_conn.register(file_name, df)
                        success = False
                        while not success:
                            Nonambigioussql_instance = Nonambigioussql(df, file_name)
                            Nonambigioussql_instance_result = Nonambigioussql_instance.Generatsql(text=goal)
                            sql = Nonambigioussql_instance_result
                            try:
                                duckdb_check_result = duckdb_conn.execute(sql).df()
                                success = True
                            except duckdb.Error as e:
                                print(f"SQL执行错误: {e}")
                        break

                else:
                    print("✓ 校验通过")
                    break
        except Exception as e:
            print(f"读取 {dataset_name} 出错: {e}")
    else:
        print(f"文件 {dataset_name} 不存在！")




=== 第 1 行 ===
问题: What kinds of movies earn the most these days?
数据集文件: Movies.csv
成功读取 Movies.csv, 行数: 709 列数: 10
                 Title  Worldwide Gross  Production Budget  Release Year  \
0  From Dusk Till Dawn         25728961           20000000          1996   
1         Broken Arrow        148345997           65000000          1996   
2            City Hall         20278055           40000000          1996   
3        Happy Gilmore         38623460           10000000          1996   
4                Fargo         51204567            7000000          1996   

  Content Rating  Running Time     Genre         Creative Type  \
0              R           107    Horror               Fantasy   
1              R           108    Action  Contemporary Fiction   
2              R           111     Drama  Contemporary Fiction   
3          PG-13            92    Comedy  Contemporary Fiction   
4              R            87  Thriller  Contemporary Fiction   

   Rotten Tomatoes Rating  IMD

KeyboardInterrupt: 